<a href="https://colab.research.google.com/github/foultraengineer9/AI_Final_Project/blob/main/Final_Story_library_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk spacy requests
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 41.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import en_core_web_md

# Load directly without needing a runtime restart
nlp = en_core_web_md.load()

In [3]:
!pip install cefrpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.6/888.6 kB 13.1 MB/s eta 0:00:00


In [4]:
import json
import re
import requests
import nltk
import spacy
from cefrpy import CEFRAnalyzer
from nltk.corpus import wordnet as wn

# INITIALIZATION & SETUP
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    import en_core_web_md
    nlp = en_core_web_md.load()

cefr_analyzer = CEFRAnalyzer()
CEFR_NUMERIC_MAP = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}

# Hardcoded Backup Dataset (50 passages available)
FALLBACK_STORIES = [
    {"title": "Outer Space", "text": "Space exploration uses astronomy and technology to explore outer space. Physical exploration is conducted by human spaceflights and robotic spacecraft. Massive rockets push payloads beyond Earth gravity."},
    {"title": "Ancient Egypt", "text": "Ancient Egypt was a civilization in North Africa along the lower reaches of the Nile River. Its success came from its ability to adapt to agriculture. The flooding provided rich soil for gigantic harvests."},
    {"title": "Bicycle", "text": "A bicycle is a human-powered land vehicle with two wheels attached to a frame. Introduced in the 19th century, riding a bicycle is an efficient form of transportation that promotes physical fitness."},
    {"title": "Ocean Deep", "text": "The deep ocean is the lowest layer in the ocean, existing below the thermocline. Immense pressure and cold temperatures make this environment harsh for standard marine life."},
    {"title": "Volcanoes", "text": "A volcano is a rupture in the crust of a planetary-mass object that allows hot lava and volcanic ash to escape from a magma chamber below the surface."},
    {"title": "Dinosaur", "text": "Dinosaurs are a diverse group of reptiles of the clade Dinosauria. They first appeared during the Triassic period and became the dominant terrestrial vertebrates."},
    {"title": "Electricity", "text": "Electricity is the set of physical phenomena associated with the presence and motion of matter that has a property of electric charge."},
    {"title": "Renaissance", "text": "The Renaissance was a fervent period of European cultural, artistic, political, and economic rebirth following the Middle Ages."},
    {"title": "Photosynthesis", "text": "Photosynthesis is a biological process used by plants to convert light energy into chemical energy that can later be released to fuel activities."},
    {"title": "Pyramid", "text": "A pyramid is a structure whose outer surfaces are triangular and converge to a single point at the top. The base of a pyramid can be trilateral or quadrilateral."},
    {"title": "Glacier", "text": "A glacier is a persistent body of dense ice that is constantly moving under its own weight. It forms where accumulation of snow exceeds its ablation."},
    {"title": "Atmosphere", "text": "An atmosphere is a layer of gas that surrounds a body of matter. Gases are retained by the gravity of the body if gravity is high and temperature is low."},
    {"title": "Architecture", "text": "Architecture is both the process and the product of planning, designing, and constructing buildings or other structures."},
    {"title": "Biodiversity", "text": "Biodiversity is the variety and variability of life on Earth. It is a measure of variation at the genetic, species, and ecosystem levels."},
    {"title": "Tsunami", "text": "A tsunami is a series of waves in a water body caused by the displacement of a large volume of water, generally in an ocean or a large lake."},
    {"title": "Solar System", "text": "The Solar System is the gravitationally bound system of the Sun and the objects that orbit it, either directly or indirectly."},
    {"title": "Microscope", "text": "A microscope is a laboratory instrument used to examine objects that are too small to be seen by the naked eye."},
    {"title": "Gravity", "text": "Gravity is a fundamental interaction which causes mutual attraction between all things with mass or energy."},
    {"title": "Fossil", "text": "A fossil is any preserved remains, impression, or trace of any once-living thing from a past geological age."},
    {"title": "Antibiotics", "text": "Antibiotics are medications that destroy or slow down the growth of bacteria to treat infections in humans and animals."},
    {"title": "Climate Change", "text": "Climate change includes both global warming driven by human-induced emissions of greenhouse gases and the resulting large-scale shifts in weather patterns."},
    {"title": "DNA Structure", "text": "Deoxyribonucleic acid is a polymer composed of two polynucleotide chains that coil around each other to form a double helix carrying genetic instructions."},
    {"title": "Ecosystems", "text": "An ecosystem consists of all the organisms and the physical environment with which they interact within a designated geographical region."},
    {"title": "Plate Tectonics", "text": "Plate tectonics is a scientific theory describing the large-scale motion of seven large plates and the movements of a larger number of smaller plates of Earth's lithosphere."},
    {"title": "Renewable Energy", "text": "Renewable energy is energy derived from natural resources that replenish themselves in less time than a human lifetime without depleting global resources."},
    {"title": "Artificial Intelligence", "text": "Artificial intelligence is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans."},
    {"title": "Telescope", "text": "A telescope is an optical instrument that makes distant objects appear magnified by using an arrangement of lenses or curved mirrors."},
    {"title": "Microbiology", "text": "Microbiology is the scientific study of microorganisms, those being unicellular, multicellular, or acellular living microscopic structures."},
    {"title": "Vaccines", "text": "A vaccine is a biological preparation that provides active acquired immunity to a particular infectious disease by stimulating antibodies."},
    {"title": "Black Holes", "text": "A black hole is a region of spacetime where gravity is so strong that nothing, including light and other electromagnetic waves, has enough energy to escape."},
    {"title": "Genetics", "text": "Genetics is a branch of biology concerned with the study of genes, genetic variation, and heredity in organisms."},
    {"title": "Geology", "text": "Geology is a branch of Earth science concerned with the solid Earth, the rocks of which it is composed, and the processes by which they change over time."},
    {"title": "Mathematics", "text": "Mathematics includes the study of such topics as numbers, formulas and related structures, shapes and spaces in which they are contained, and quantities and their changes."},
    {"title": "Philosophy", "text": "Philosophy is the systematized study of general and fundamental questions touching upon topics such as existence, reason, knowledge, value, mind, and language."},
    {"title": "Economics", "text": "Economics is the social science that studies the production, distribution, and consumption of goods and services across different global markets."},
    {"title": "Psychology", "text": "Psychology is the scientific study of mind and behavior, encompassing the biological pressures, social pressures, and environmental factors that influence actions."},
    {"title": "Sociology", "text": "Sociology is a social science that focuses on society, human social behavior, patterns of social relationships, social interaction, and aspects of culture."},
    {"title": "Anthropology", "text": "Anthropology is the scientific study of humanity, concerned with human behavior, human biology, cultures, societies, and linguistics in both the present and past."},
    {"title": "Botany", "text": "Botany is the branch of biology that deals with the study of plants, including their structure, properties, and biochemical processes."},
    {"title": "Zoology", "text": "Zoology is the branch of biology that studies the animal kingdom, including the structure, embryology, evolution, classification, habits, and distribution of all animals."},
    {"title": "Astronomy", "text": "Astronomy is a natural science that studies celestial objects and phenomena, using mathematics, physics, and chemistry in order to explain their origin and evolution."},
    {"title": "Thermodynamics", "text": "Thermodynamics is a branch of physics that deals with heat, work, and temperature, and their relation to energy, entropy, and physical properties of matter."},
    {"title": "Quantum Mechanics", "text": "Quantum mechanics is a fundamental theory in physics that provides a description of the physical properties of nature at the scale of atoms and subatomic particles."},
    {"title": "Nanotechnology", "text": "Nanotechnology is the engineering of functional systems at the molecular scale, covering a broad range of topics in physics, chemistry, and materials science."},
    {"title": "Robotics", "text": "Robotics is an interdisciplinary branch of engineering and computer science that involves the design, construction, operation, and use of robots."},
    {"title": "Cybersecurity", "text": "Cybersecurity is the practice of protecting systems, networks, and programs from digital attacks aimed at accessing, changing, or destroying sensitive information."},
    {"title": "Cloud Computing", "text": "Cloud computing is the on-demand availability of computer system resources, especially data storage and computing power, without direct active management by the user."},
    {"title": "Machine Learning", "text": "Machine learning is a field of study devoted to understanding and building methods that learn from data to improve performance on a set of tasks."},
    {"title": "Biotechnology", "text": "Biotechnology is the integration of natural sciences and engineering sciences in order to achieve the application of organisms, cells, parts thereof and molecular analogues for products."},
    {"title": "Neuroscience", "text": "Neuroscience is the scientific study of the nervous system, combining physiology, anatomy, molecular biology, developmental biology, cytology, computer science, and mathematical modeling."}
]

# 1. EXPANDED WIKIPEDIA SCRAPER (60 Topics Target for 50 Final Stories)
def fetch_50_wikipedia_stories():
    target_count = 50
    topics = [
        "Outer space", "Ancient Egypt", "Bicycle", "Ocean", "Volcano",
        "Dinosaur", "Electricity", "Renaissance", "Photosynthesis", "Pyramid",
        "Glacier", "Atmosphere", "Architecture", "Biodiversity", "Tsunami",
        "Solar system", "Microscope", "Gravity", "Fossil", "Antibiotic",
        "Magnetism", "Ecosystem", "DNA", "Tornado", "Galileo Galilei",
        "Comet", "Climate", "Evolution", "Supernova", "Telescope",
        "Plate tectonics", "Renewable energy", "Black hole", "Vaccine", "Algorithm",
        "Geography", "Mathematics", "Botany", "Philosophy", "Economics",
        "Psychology", "Sociology", "Anthropology", "Zoology", "Astronomy",
        "Thermodynamics", "Quantum mechanics", "Nanotechnology", "Robotics", "Cybersecurity",
        "Cloud computing", "Machine learning", "Biotechnology", "Neuroscience", "Genetics",
        "Geology", "Meteorology", "Linguistics", "Archaeology", "Optics"
    ]

    stories = []
    base_url = "https://simple.wikipedia.org/w/api.php"
    headers = {"User-Agent": "ContentLibrarianBot/3.0 (educational_project)"}

    print(f"Fetching up to {target_count} passages from Simple Wikipedia...")
    for title in topics:
        if len(stories) >= target_count:
            break

        params = {
            "action": "query", "format": "json", "titles": title,
            "prop": "extracts", "exintro": True, "explaintext": True
        }
        try:
            res = requests.get(base_url, params=params, headers=headers, timeout=3).json()
            pages = res.get("query", {}).get("pages", {})
            for page_id, page_info in pages.items():
                text = page_info.get("extract", "").strip()
                text = re.sub(r'\[\d+\]', '', text)
                text = re.sub(r'\s+', ' ', text)

                if len(text) >= 50:
                    stories.append({"title": title, "text": text})
                    print(f"  [✓] Wikipedia API loaded: '{title}'")
        except Exception:
            continue

    # SMART FILL: Append fallback stories if Wikipedia returns fewer than 50
    if len(stories) < target_count:
        needed = target_count - len(stories)
        print(f"\n[!] Wikipedia fetched {len(stories)} stories. Appending {needed} fallback stories to reach exactly 50...")

        existing_titles = {s["title"].lower() for s in stories}
        for fallback in FALLBACK_STORIES:
            if len(stories) >= target_count:
                break
            if fallback["title"].lower() not in existing_titles:
                stories.append(fallback)

    return stories

# 2. LEXICAL & SEMANTIC SYNONYM ENGINE
def get_cefr_numeric_level(word):
    word_str = word.lower()
    try:
        res = cefr_analyzer.find_word(word_str)
        if res:
            level = str(res[0] if isinstance(res, (list, tuple)) else res).upper()
            if level in CEFR_NUMERIC_MAP:
                score = CEFR_NUMERIC_MAP[level]
                if score >= 3:  # B1 or higher
                    return score
    except Exception:
        pass

    if len(word_str) >= 6:
        return 3

    return None

def find_best_semantic_synonym(token):
    word_str = token.text.lower()
    spacy_word = nlp(word_str)

    if not spacy_word.has_vector or token.is_stop or len(word_str) <= 3:
        return None

    candidate_synonyms = set()
    for synset in wn.synsets(word_str):
        for lemma in synset.lemmas():
            candidate = lemma.name().replace("_", " ").lower()
            if candidate != word_str and " " not in candidate:
                candidate_synonyms.add(candidate)

    if not candidate_synonyms:
        return None

    best_synonym = None
    best_similarity = -1.0

    for candidate in candidate_synonyms:
        cand_doc = nlp(candidate)
        if cand_doc.has_vector:
            similarity = spacy_word.similarity(cand_doc)
            if similarity > best_similarity:
                best_similarity = similarity
                best_synonym = candidate

    if best_similarity >= 0.50:
        return best_synonym
    return None

def process_passage(story_data):
    doc = nlp(story_data["text"])
    synonym_map = {}

    for token in doc:
        if token.pos_ in {"NOUN", "VERB", "ADJ", "ADV"} and token.is_alpha:
            target_word = token.text.lower()
            if target_word not in synonym_map:
                if get_cefr_numeric_level(target_word):
                    best_synonym = find_best_semantic_synonym(token)
                    if best_synonym:
                        synonym_map[target_word] = best_synonym

    return {
        "title": story_data["title"],
        "text": story_data["text"],
        "synonyms": synonym_map
    }

# ---------------------------------------------------------------------------
# 3. RUNNER & EXPORTER
# ---------------------------------------------------------------------------
stories = fetch_50_wikipedia_stories()
library = [process_passage(s) for s in stories]

with open("stories_library.json", "w", encoding="utf-8") as f:
    json.dump(library, f, indent=2, ensure_ascii=False)

print(f"\n==========================================")
print(f"SUCCESS: Generated 'stories_library.json' with EXACTLY {len(library)} stories!")
print(f"==========================================")

Fetching up to 50 passages from Simple Wikipedia...
  [✓] Wikipedia API loaded: 'Outer space'
  [✓] Wikipedia API loaded: 'Ancient Egypt'
  [✓] Wikipedia API loaded: 'Bicycle'
  [✓] Wikipedia API loaded: 'Ocean'
  [✓] Wikipedia API loaded: 'Volcano'
  [✓] Wikipedia API loaded: 'Dinosaur'
  [✓] Wikipedia API loaded: 'Electricity'
  [✓] Wikipedia API loaded: 'Renaissance'
  [✓] Wikipedia API loaded: 'Photosynthesis'
  [✓] Wikipedia API loaded: 'Pyramid'

[!] Wikipedia fetched 10 stories. Appending 40 fallback stories to reach exactly 50...

SUCCESS: Generated 'stories_library.json' with EXACTLY 50 stories!


In [5]:
import json

with open("stories_library.json", "r") as f:
    data = json.load(f)

print(f"Total Stories Outputted: {len(data)}\n" + "="*40)
for idx, item in enumerate(data, 1):
    print(f"[{idx}] {item['title']} - Mapped {len(item['synonyms'])} synonyms")

Total Stories Outputted: 50
[1] Outer space - Mapped 9 synonyms
[2] Ancient Egypt - Mapped 12 synonyms
[3] Bicycle - Mapped 10 synonyms
[4] Ocean - Mapped 17 synonyms
[5] Volcano - Mapped 24 synonyms
[6] Dinosaur - Mapped 31 synonyms
[7] Electricity - Mapped 28 synonyms
[8] Renaissance - Mapped 18 synonyms
[9] Photosynthesis - Mapped 10 synonyms
[10] Pyramid - Mapped 4 synonyms
[11] Ocean Deep - Mapped 3 synonyms
[12] Volcanoes - Mapped 2 synonyms
[13] Glacier - Mapped 3 synonyms
[14] Atmosphere - Mapped 1 synonyms
[15] Architecture - Mapped 3 synonyms
[16] Biodiversity - Mapped 6 synonyms
[17] Tsunami - Mapped 2 synonyms
[18] Solar System - Mapped 2 synonyms
[19] Microscope - Mapped 2 synonyms
[20] Gravity - Mapped 4 synonyms
[21] Fossil - Mapped 5 synonyms
[22] Antibiotics - Mapped 8 synonyms
[23] Climate Change - Mapped 4 synonyms
[24] DNA Structure - Mapped 2 synonyms
[25] Ecosystems - Mapped 4 synonyms
[26] Plate Tectonics - Mapped 4 synonyms
[27] Renewable Energy - Mapped 1 synon